# Notebook 42 — Balanced SupCon + class-conditional routing

This notebook completes the controlled 2 × 2 representation-by-routing comparison. The candidate reuses the exact Balanced-SupCon Stage-A encoder and Stage-B linear experts, freezes both throughout Stage C, retains the dropout-free `64 → 32 → 4` ReLU gate and `1e-3` Stage-C learning rate, and adds only the zero-initialized `4 experts × 22 classes` reliability matrix.

The candidate therefore trains only 2,300 Stage-C parameters: 2,212 gate parameters plus 88 reliability parameters. It does not retrain or modify Balanced SupCon.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
CE_STANDARD_PREFIX = 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'
CE_CLASS_CONDITIONAL_PREFIX = 'nfv3_4way_moe_soft_deeper_gate32_class_conditional_frozen_stagec_v1'
REFERENCE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_deeper_gate32_frozen_stagec_v1'
REFERENCE_CONFIG = 'config/compact_soft_moe_balanced_supcon_frozen_deeper_gate_3seed.yaml'
CANDIDATE_PREFIX = 'nfv3_4way_moe_soft_balanced_supcon_class_conditional_deeper_gate32_frozen_stagec_v1'
CANDIDATE_CONFIG = 'config/compact_soft_moe_balanced_supcon_class_conditional_frozen_deeper_gate_3seed.yaml'
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS = 2  # Seeds 0–1 currently have Balanced-SupCon A/B artifacts; confirm seed 2 later.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight and Stage-C-only execution

The preflight proves that the candidate and Balanced-SupCon reference differ only in class-conditional routing and its `1e-4` reliability regularizer. Both candidate reuse fields point to the signed Balanced-SupCon artifacts, so only Stage C is scheduled.

In [ ]:
import copy, torch, yaml
from models.encoder import build_encoder
from training.compact_soft_moe_run import CompactSoftMoEStudy
from training.logging_utils import run_streaming_logged
from training.model_utils import apply_stage_c_trainability, build_model
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
reference = yaml.safe_load(Path(REFERENCE_CONFIG).read_text())
candidate = yaml.safe_load(Path(CANDIDATE_CONFIG).read_text())
assert reference['seeds'] == candidate['seeds'] == [0, 1, 2]
assert candidate['reuse_stage_a_from_prefix'] == candidate['reuse_stage_b_from_prefix'] == REFERENCE_PREFIX
ref = copy.deepcopy(reference['backbone']); cand = copy.deepcopy(candidate['backbone'])
assert ref.pop('architecture') == 'moe_dataset_soft'
assert cand.pop('architecture') == 'moe_dataset_class_conditional'
assert cand['training']['stage_c'].pop('lambda_reliability') == 0.0001
assert ref == cand, 'Only class-conditional routing/reliability may differ from Balanced SupCon'
assert candidate['backbone']['training']['representation']['objective'] == 'balanced_supcon'
assert candidate['backbone']['training']['representation']['weight'] == 0.1
assert candidate['backbone']['training']['stage_c_unfreeze'] == 'none'
assert candidate['backbone']['training']['stage_c']['freeze_experts'] is True
assert candidate['backbone']['training']['stage_c']['optimizer']['lr'] == 0.001
model_cfg = candidate['backbone']['model']
model = build_model('moe_dataset_class_conditional', build_encoder(47, 64, model_cfg['encoder']), ['UNSW', 'ToN', 'BoT', 'CIC'], [f'class_{i}' for i in range(22)], model_cfg)
apply_stage_c_trainability(model, candidate['backbone'])
assert model.class_reliability.shape == (4, 22)
assert torch.count_nonzero(model.class_reliability).item() == 0
assert sum(p.numel() for p in model.parameters()) == candidate['expected_total_parameters'] == 22420
assert sum(p.numel() for p in model.parameters() if p.requires_grad) == 2300
runner_check = CompactSoftMoEStudy(base_config_path='config/default.yaml', study_config_path=CANDIDATE_CONFIG, prefix=CANDIDATE_PREFIX, execute=False)
assert runner_check._final_nested(0)['training']['stages'] == ['C']
if RUN_TESTS: subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', CANDIDATE_CONFIG, '--prefix', CANDIDATE_PREFIX]
run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label='42-bsupcon-class-conditional-dry-run')
if EXECUTE: run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label='42-bsupcon-class-conditional-training')
else: print('NO TRAINING WAS STARTED. Review the contract, set EXECUTE=True, and rerun this cell.')

## Validation-first 2 × 2 factorial comparison

The four cells are CE/standard routing, CE/class-conditional routing, Balanced-SupCon/standard routing, and Balanced-SupCon/class-conditional routing. All comparisons use only seeds completed by every required condition. The interaction term tests whether the class-conditional gain changes specifically under Balanced SupCon rather than merely repeating its CE effect.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
conditions = {
    'ce_standard': CE_STANDARD_PREFIX,
    'ce_class_conditional': CE_CLASS_CONDITIONAL_PREFIX,
    'bsupcon_standard': REFERENCE_PREFIX,
    'bsupcon_class_conditional': CANDIDATE_PREFIX,
}
factor = {
    'ce_standard': ('CE', 'standard'), 'ce_class_conditional': ('CE', 'class_conditional'),
    'bsupcon_standard': ('BalancedSupCon', 'standard'), 'bsupcon_class_conditional': ('BalancedSupCon', 'class_conditional'),
}
validation_rows = []; test_rows = []; class_rows = []; dataset_rows = []; resource_rows = []
for condition, prefix in conditions.items():
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}'
        required = ['manifest.json', 'Validation_Overall_Metrics.csv', 'Validation_Per_Dataset_Metrics.csv', 'Validation_Per_Class_Metrics.csv', 'Overall_Metrics.csv', 'Per_Class_Metrics.csv', 'Per_Dataset_Metrics.csv', 'Resource_Accounting.csv']
        if not all((result_dir / name).is_file() for name in required): continue
        overall = pd.read_csv(result_dir / 'Validation_Overall_Metrics.csv').iloc[0]
        per_dataset = pd.read_csv(result_dir / 'Validation_Per_Dataset_Metrics.csv')
        per_class = pd.read_csv(result_dir / 'Validation_Per_Class_Metrics.csv'); present = per_class['support'] > 0
        rare = per_class[(per_class['support'] >= 20) & (per_class['support'] < 200)]
        representation, routing = factor[condition]
        validation_rows.append({'condition': condition, 'representation': representation, 'routing': routing, 'seed': seed, 'validation_macro_f1': overall['macro_f1'], 'validation_accuracy': overall['accuracy'], 'validation_weighted_f1': np.average(per_class.loc[present, 'f1'], weights=per_class.loc[present, 'support']), 'validation_macro_precision': overall['macro_precision'], 'validation_macro_recall': overall['macro_recall'], 'worst_dataset_macro_f1': per_dataset['macro_f1'].min(), 'rare_recall_mean': rare['recall'].mean() if len(rare) else np.nan, 'weakest_class_recall': per_class.loc[present, 'recall'].min()})
        locked = pd.read_csv(result_dir / 'Overall_Metrics.csv').query("origin == 'ALL'").copy(); locked.insert(0, 'seed', seed); locked.insert(0, 'condition', condition); test_rows.append(locked)
        classes = pd.read_csv(result_dir / 'Per_Class_Metrics.csv').query("origin == 'ALL'").copy(); classes.insert(0, 'seed', seed); classes.insert(0, 'condition', condition); class_rows.append(classes)
        datasets = pd.read_csv(result_dir / 'Per_Dataset_Metrics.csv').copy(); datasets.insert(0, 'seed', seed); datasets.insert(0, 'condition', condition); dataset_rows.append(datasets)
        resources = pd.read_csv(result_dir / 'Resource_Accounting.csv').copy(); resources.insert(0, 'seed', seed); resources.insert(0, 'condition', condition); resource_rows.append(resources)
validation = pd.DataFrame(validation_rows)
if validation.empty: print('No completed factorial cells exist yet.')
else:
    seed_sets = [set(validation.query('condition == @name')['seed']) for name in conditions]
    common = sorted(set.intersection(*seed_sets)) if seed_sets else []
    print('Common paired seeds across all four cells:', common)
    if len(common) < 2: print('EXPLORATORY ONLY: fewer than two complete factorial seeds.')
    paired = validation[validation['seed'].isin(common)].copy(); display(paired)
    metrics = ['validation_macro_f1', 'validation_accuracy', 'validation_weighted_f1', 'validation_macro_precision', 'validation_macro_recall', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class_recall']
    for metric in metrics:
        pivot = paired.pivot(index='seed', columns='condition', values=metric)
        pivot['candidate_minus_bsupcon_standard'] = pivot['bsupcon_class_conditional'] - pivot['bsupcon_standard']
        pivot['candidate_minus_ce_class_conditional'] = pivot['bsupcon_class_conditional'] - pivot['ce_class_conditional']
        pivot['factorial_interaction'] = (pivot['bsupcon_class_conditional'] - pivot['bsupcon_standard']) - (pivot['ce_class_conditional'] - pivot['ce_standard'])
        print(metric); display(pivot); print('Mean candidate gain over Balanced SupCon:', pivot['candidate_minus_bsupcon_standard'].mean(), 'interaction:', pivot['factorial_interaction'].mean())
    if common and test_rows:
        test = pd.concat(test_rows, ignore_index=True).query('seed in @common')
        test_metrics = [c for c in ['accuracy', 'micro_f1', 'weighted_f1', 'macro_f1', 'macro_precision', 'macro_recall', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
        print('=== Locked-test descriptive metrics; never used for selection ==='); display(test.groupby('condition')[test_metrics].agg(['mean', 'std']))
        print('=== Per-dataset test macro-F1 ==='); display(pd.concat(dataset_rows, ignore_index=True).query('seed in @common').pivot_table(index='dataset', columns='condition', values='macro_f1', aggfunc='mean'))
        print('=== Per-class test F1 ==='); display(pd.concat(class_rows, ignore_index=True).query('seed in @common').pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    if resource_rows:
        resources = pd.concat(resource_rows, ignore_index=True); display(resources)
        candidate_resources = resources.query("condition == 'bsupcon_class_conditional'")
        assert set(candidate_resources['total_parameters'].astype(int)) == {22420}
        assert set(candidate_resources['trainable_parameters'].astype(int)) == {2300}

## Reused-body integrity and class-conditional routing diagnostics

In [ ]:
for seed in (0, 1, 2):
    result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{CANDIDATE_PREFIX}_seed{seed}'
    if not (result_dir / 'manifest.json').is_file(): continue
    source_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{REFERENCE_PREFIX}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
    candidate_c = pd.read_csv(result_dir / 'Validation_Expert_Owned_StageC.csv')
    preserved = source_b.merge(candidate_c, on='dataset', suffixes=('_stage_b', '_stage_c')); delta_columns = []
    for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
        column = f'delta_{metric}'; preserved[column] = preserved[f'{metric}_stage_c'] - preserved[f'{metric}_stage_b']; delta_columns.append(column)
    assert np.allclose(preserved[delta_columns], 0.0, atol=1e-12), f'Reused Balanced-SupCon body drift in seed {seed}'
    print(f'=== Seed {seed}: exact Balanced-SupCon expert preservation ==='); display(preserved)
    print(f'=== Seed {seed}: gate entropy/confidence by dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
    print(f'=== Seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))
    print(f'=== Seed {seed}: learned expert × class reliability ==='); display(pd.read_csv(result_dir / 'Class_Conditional_Reliability.csv'))
    history_path = result_dir / 'training' / 'Training_History.csv'
    if history_path.is_file():
        history = pd.read_csv(history_path).query("stage == 'C'")
        columns = [c for c in ['epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'train_reliability_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left'] if c in history]
        print(f'=== Seed {seed}: Stage-C history ==='); display(history[columns])

## Interpretation rule

Call the combination successful only if its paired validation macro-F1 exceeds Balanced SupCon, the direction is consistent across seeds, and accuracy, worst-dataset macro-F1, and rare-class recall do not show a material compensating regression. The factorial interaction distinguishes genuine complementarity from simply reproducing the class-conditional effect already observed with CE. Locked-test metrics support interpretation but never select the model. A three-seed `Final_Study_Manifest.json` is produced only after Balanced-SupCon seed 2 exists and the candidate's seed 2 is run.